<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Visualizing trends - Solution

---

### About
Understand how to create charts within `pandas` using the `.plot()` method.

### Learning Objectives
- Chart dates in `pandas`
- Create line charts with `matplotlib`

### Notebook Guide
- Scenario
- Working with dates in `pandas`
- Now You Try!
- Conclusions and Takeaways

# Scenario

You're an analyst working for CoolBricks - a new toy company who wants to compete with LEGO by producing their own bricks and sets. To better understand what kind of sets to create and at what price points, you are tasked with analyzing LEGO's catalogue over time.

The data comes from Kaggle: [https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time](https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time)

We are going to use `pandas` to read this data and `matplotlib` to create plots.

Our research question is: ***How have the number of products changed over time?***

# Working with dates in `pandas`
---

In [ ]:
import pandas as pd

Let's look at the data

In [ ]:
lego = pd.read_csv("../data/lego.csv")
lego.head()

We have a "year" column which is an integer, but it could also be a date!

To convert a string to a date we can use the `pandas` `.to_datetime()` method, where we can also specify the format of the date. *See [http://strftime.org/](http://strftime.org/) for more details about datetime formats in Python.*

If we don't specify the format, we get this:

In [ ]:
pd.to_datetime(lego["Year"])

Let's tell `pandas` that the string actually represents the year (and `pandas` will come up with defaults for the month and day)

In [ ]:
pd.to_datetime(lego["Year"], format="%Y")

Great, let's save the "date version" as a new column

In [ ]:
lego["year_date"] = pd.to_datetime(lego["Year"], format="%Y")
lego.head()

Now, we can access date components with the `.dt` accessor

In [ ]:
lego["year_date"].dt.year

And we have two ways of calculating "products per year". The first is to use the original `Year` column:

In [ ]:
lego.groupby("Year").size()

Or we can set the `index` of the `DataFrame` to the date column and use `resample` (which is like `groupby` but specifically for dates)

In [ ]:
lego.set_index("year_date").head()

In [ ]:
annual_count = (
    lego.set_index("year_date")
    .resample("YS")  # "groupby" year
    .size()  # count the rows
)

annual_count

The advantage of `resample` is we get a `Series` that already has a temporal index. This helps with creating line charts with better defaults, for example.

To create a line chart from this data, we can simply call `.plot()` (the default type is a line chart)

In [ ]:
annual_count.plot();

As with most plots, we can set some additional values

In [ ]:
annual_count.plot(
    title="LEGO have generally released more products per year over time",
    xlabel="Year",
    ylabel="Number of products",
);

# Now You Try!
----

1. Create a line chart showing the _**average product retail price**_ over time.

What do you conclude?

How can you improve the chart?

In [ ]:
annual_price = lego.set_index("year_date").resample("YS")["USD_MSRP"].median()
annual_price.plot();

Looks like there is a lot of missing data before the year 2000. We could use that as a cutoff and only visualize data after that year.

In [ ]:
annual_price = (
    lego[lego["Year"] >= 2000]
    .set_index("year_date")
    .resample("YS")["USD_MSRP"]
    .median()
)

annual_price.plot(
    title="LEGO products have increased in price over time (which might just be inflation)",
    xlabel="Year",
    ylabel="Median product price ($)",
);

2. *BONUS:* Create a new column to calculate the _**decade**_ a product was released. Then calculate the _**average number of pieces per product**_ per decade and plot it as a line chart.

In [ ]:
# floordiv(10) is the same as //10 which gives the integer value of a division
# so for 1978, floordiv(10) returns 197, and multiplying that by 10 gives us the decade
lego["decade"] = lego["Year"].floordiv(10).mul(10)

(
    lego.groupby("decade")["Pieces"]
    .median()  # average number of pieces per decade
    .plot(
        title="Products contain more pieces on average as the decades go on",
        xlabel="Decade",
        ylabel="Median number of pieces per product",
    )
);

# Conclusions and Takeaways

- `pandas` lets us convert strings to dates with `to_datetime()`
- Once a column is a date, more functionality is available to us
- One use case of date columns is the easier creation of line charts